# 🧠 05. Kaggle Notebook: LSTM Trajectory Classifier Training
Huấn luyện mạng Bidirectional LSTM + Attention phân loại chuỗi động học 15 đặc trưng thành xác suất tai nạn.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from src.classifiers.lstm_classifier import TrajectoryAccidentClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Dataset đặc trưng động học

In [ ]:
feat_path = "datasets/features/extracted_features.npz"
if os.path.exists(feat_path):
    data = np.load(feat_path)
    X = data["X"]
    y = data["y"]
else:
    print("Generating synthetic training batch for demonstration...")
    X = np.random.randn(500, 30, 15).astype(np.float32)
    y = np.random.choice([0, 1], size=(500,), p=[0.75, 0.25]).astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
train_loader = DataLoader(TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float().unsqueeze(1)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val).float().unsqueeze(1)), batch_size=32, shuffle=False)
print(f"Train samples: {len(X_train)}, Val samples: {len(X_val)}")

## 2. Huấn luyện với Focal Loss

In [ ]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.bce = nn.BCELoss(reduction='none')

    def forward(self, pred, target):
        bce = self.bce(pred, target)
        p_t = pred * target + (1 - pred) * (1 - target)
        loss = self.alpha * ((1 - p_t) ** self.gamma) * bce
        return loss.mean()

model = TrajectoryAccidentClassifier(input_dim=15, hidden_dim=64, num_layers=2).to(device)
criterion = BinaryFocalLoss(gamma=2.0, alpha=0.75)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(15):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        preds = model(bx)
        loss = criterion(preds, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/15] Loss: {total_loss / len(train_loader):.4f}")

model.save_checkpoint("checkpoints/lstm_trajectory_best.pt")
print("✅ Saved LSTM best checkpoint!")